In [4]:
import os
import random
import json
import time
import glob
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load environment variables
load_dotenv()

api_key = os.getenv("GOOGLE_AI_STUDIO")
if not api_key:
    raise ValueError("API Key not found. Please check your .env file.")

client = genai.Client(api_key=api_key)

# Configuration for Gemini 1.5 Flash
# Flash is ideal for high-volume, low-latency tasks like this
MODEL_NAME = "gemini-2.5-flash"

# System instruction
SYSTEM_INSTRUCTION = """
    You are an expert data entry specialist. 
    Your task is to extract information from handwritten loan application forms.
    The forms are in Spanish.
    
    IMPORTANT: The image provided might be rotated 90 degrees clockwise or counter-clockwise. 
    Analyze the text orientation visually and extract the data correctly regardless of the image rotation.
    
    Extract data into three specific sections:
    1. SOLICITANTE
    2. CODEUDOR (First one found)
    3. CODEUDOR (Second one found, if applicable)
    
    If any information is illegible or cannot be found, set the value to blank.
    
    Return the output strictly as a JSON object.
    """

print("Setup complete. Client configured.")


Setup complete. Client configured.


In [2]:
def save_json(data, original_filename):
    """Saves the extracted data to a JSON file."""
    output_filename = f"{Path(original_filename).stem}_extracted.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"Saved: {output_filename}")


In [3]:
# 1. Locate files
clients_folder = "clients"
pdf_files = glob.glob(os.path.join(clients_folder, "*.pdf"))

if not pdf_files:
    print("No PDF files found in the 'clients' folder.")
else:
    # 2. Select 10 random files (or fewer if less than 10 exist)
    num_files = min(5, len(pdf_files))
    selected_files = random.sample(pdf_files, num_files)

    print(f"Selected {len(selected_files)} files for processing.\n")

    # 3. Process each file
    for i, pdf_path in enumerate(selected_files):
        print(f"[{i+1}/{len(selected_files)}] Processing: {pdf_path}")
        
        try:
            # Upload the PDF directly to Gemini
            # We use a binary stream and a safe display name to avoid UnicodeEncodeError on Windows
            with open(pdf_path, "rb") as f:
                pdf_file = client.files.upload(
                    file=f,
                    config=types.UploadFileConfig(
                        display_name="loan_document.pdf",
                        mime_type="application/pdf"
                    )
                )
            
            # Prompt specifically for the second page
            prompt = """
            Analyze the SECOND page of this document.
            Extract all handwritten fields from the form on that page into the defined JSON structure.
            """
            
            # Generate content
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[pdf_file, prompt],
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_INSTRUCTION,
                    temperature=0.1,
                    response_mime_type="application/json"
                )
            )
            
            # Parse response
            json_data = json.loads(response.text)
            
            # Add metadata for traceability
            json_data['meta_source_file'] = os.path.basename(pdf_path)
            json_data['meta_processed_at'] = time.strftime("%Y-%m-%d %H:%M:%S")
            
            # Save to disk
            save_json(json_data, pdf_path)
            
            # Cleanup: Delete the file from Gemini to avoid clutter
            client.files.delete(name=pdf_file.name)
            
        except Exception as e:
            print(f"Failed to extract data from {pdf_path}: {e}")
        
        # Rate Limiting for Free Tier
        # Free tier allows ~15 RPM (Requests Per Minute). 
        # A 4-second sleep ensures we stay safe.
        time.sleep(4) 

    print("\nProcessing complete.")


Selected 5 files for processing.

[1/5] Processing: clients\20251210 Credito ordinario  Alvarez Perez, Clemencia 1.pdf
Saved: 20251210 Credito ordinario  Alvarez Perez, Clemencia 1_extracted.json
[2/5] Processing: clients\20250403 Credito ordinario Amaya Peña, Yessica Alejandra 1.pdf
Failed to extract data from clients\20250403 Credito ordinario Amaya Peña, Yessica Alejandra 1.pdf: 'ascii' codec can't encode character '\xf1' in position 35: ordinal not in range(128)
[3/5] Processing: clients\20220722 Credito ordinario Albadan Falla, Ludibia 1.pdf
Saved: 20220722 Credito ordinario Albadan Falla, Ludibia 1_extracted.json
[4/5] Processing: clients\20230809 Credito ordinario  Alvarez Perez, Clemencia 1.pdf
Saved: 20230809 Credito ordinario  Alvarez Perez, Clemencia 1_extracted.json
[5/5] Processing: clients\20230303 Credito ordinario Amaya Peña, Yessica Alejandra 1.pdf
Failed to extract data from clients\20230303 Credito ordinario Amaya Peña, Yessica Alejandra 1.pdf: 'ascii' codec can't en